# MINT final test (E5, OPV2V): results notebook

Reads **only** the two analysis summaries (`e5_v3.json` from `v3_analyze`, `e5_test.json` from the frozen v2 `e5_analyze`) and shows every pre-registered statistic, figure and check. Nothing is recomputed; the numbers are those in the files. The status banner says whether these are final-test results or development numbers. An incomplete final test stops the notebook (pre-registered rule: no AP or criteria on partial results).

To point it at the final test, change the paths in the next cell to `runs/20261024-E5v3/e5_v3.json` and `runs/20261024-E5v3/e5_test.json`.

In [ ]:
# Inputs (relative to the repository root)
V3 = 'runs/20261024-PGD/pgd_stealth_H_results.json'
V2 = 'runs/20261024-E5/H/e5a_H.json'
PREREG = 'runs/20261024-E5/preregistration_v3_2.json'


In [ ]:
import os, sys, time, platform
from IPython.display import Markdown, display
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, 'src'))
import io
import matplotlib.pyplot as plt
from IPython.display import Image
from mint.report import style, tables as T
from mint.report.data import Inputs, DEFS, DEF_NAME
from mint.report.figures import FIGURES
T0 = time.time()
inp = Inputs(os.path.join(ROOT, V3), os.path.join(ROOT, V2) if V2 else None,
             os.path.join(ROOT, PREREG) if PREREG else None, root=ROOT)
if inp.status == 'INCOMPLETE':
    raise SystemExit('STOP: the final-test analysis is flagged incomplete; no AP or criteria on partial results.')
display(Markdown(f'> **Status: {inp.status_text}**  \n> {inp.label}'))

## Provenance

In [ ]:
p = inp.provenance()
rows = [[k, f"`{v['path']}`", f"`{v['sha256'][:16]}...`"] for k, v in p['inputs'].items()]
rows += [['git HEAD', f"`{p['git_head']}`", ''], ['frozen code commit', f"`{p['frozen_code_commit']}`", ''],
         ['scenarios / frames', f'{inp.n_scenarios()} / {inp.n_frames()}', '']]
display(Markdown(T.md(['item', 'value', 'sha256'], rows)))

## Procedural checks

In [ ]:
display(Markdown(T.checks(inp)))

## Pre-registered verdict (D-64) and key secondary

In [ ]:
v = T.verdict(inp)
display(Markdown(f"**Verdict:** {v['verdict']}  \nv2 rule, global-ranking AP: {v['pass_global']}; frame-order AP: {v['pass_frame']}"))
display(Markdown(T.criteria(inp)))
display(Markdown(T.r1_minus_bray(inp)))

## Headline table

In [ ]:
display(Markdown(T.headline(inp)))
display(Markdown(T.clean_cost(inp)))

## Figures

Each figure is drawn by the same function that writes the report figures; the table of plotted values is shown under it (`rows`).

### Figure 1. Headline: mean ΔAP@0.7 over the 4 original attacks

Each defense's mean change in AP@0.7 against no defense, over spoof/remove/shift (feature) and spoof (points). Dot = point estimate; thick line = 95% CI from the scenario bootstrap (the pre-registered primary CI); thin line = 10-frame-block bootstrap. Ego-only is far off-scale and is listed in the table only.

In [ ]:
t = style.apply('light')
fn = dict((n, f) for n, _, f, _, _ in FIGURES)['01_headline']
fig, rows = fn(inp, t)
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); plt.close(fig)
display(Image(buf.getvalue()))
print(f'{len(rows)} plotted values; first rows:'); [print(r) for r in rows[:4]]

### Figure 2a. ΔAP@0.7 per condition: clean and original attacks

Every non-PGD condition × defense, both AP definitions, 95% CI from the scenario bootstrap. Ego-only in the table only.

In [ ]:
t = style.apply('light')
fn = dict((n, f) for n, _, f, _, _ in FIGURES)['02_per_family']
fig, rows = fn(inp, t)
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); plt.close(fig)
display(Image(buf.getvalue()))
print(f'{len(rows)} plotted values; first rows:'); [print(r) for r in rows[:4]]

### Figure 2b. ΔAP@0.7 per condition: optimized (PGD) attacks

As 2a for the PGD conditions, on its own scale (PGD effects are much larger than the original attacks').

In [ ]:
t = style.apply('light')
fn = dict((n, f) for n, _, f, _, _ in FIGURES)['02b_per_family_pgd']
fig, rows = fn(inp, t)
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); plt.close(fig)
display(Image(buf.getvalue()))
print(f'{len(rows)} plotted values; first rows:'); [print(r) for r in rows[:4]]

### Figure 3. Clean cost (relative AP@0.7 loss)

Relative AP@0.7 loss on clean frames. The pre-registered success rule requires B-RAY's scenario-CI high ≤ 1% under both AP definitions.

In [ ]:
t = style.apply('light')
fn = dict((n, f) for n, _, f, _, _ in FIGURES)['03_clean_cost']
fig, rows = fn(inp, t)
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); plt.close(fig)
display(Image(buf.getvalue()))
print(f'{len(rows)} plotted values; first rows:'); [print(r) for r in rows[:4]]

### Figure 4a. Key secondary: R1 − B-RAY

Paired difference R1 − B-RAY in ΔAP@0.7, the pre-registered means and each non-PGD condition. Each panel quotes the statement the frozen analysis emitted.

In [ ]:
t = style.apply('light')
fn = dict((n, f) for n, _, f, _, _ in FIGURES)['04_r1_minus_bray']
fig, rows = fn(inp, t)
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); plt.close(fig)
display(Image(buf.getvalue()))
print(f'{len(rows)} plotted values; first rows:'); [print(r) for r in rows[:4]]

### Figure 4b. R1 − B-RAY under PGD attacks

Paired difference per PGD condition (descriptive; not part of the key secondary).

In [ ]:
t = style.apply('light')
fn = dict((n, f) for n, _, f, _, _ in FIGURES)['04b_r1_minus_bray_pgd']
fig, rows = fn(inp, t)
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); plt.close(fig)
display(Image(buf.getvalue()))
print(f'{len(rows)} plotted values; first rows:'); [print(r) for r in rows[:4]]

### Figure 5. PGD attacks: recovery

Recovery = ΔAP@0.7 / attack loss per optimized-attack condition. n/a when the attack loss is < 0.005.

In [ ]:
t = style.apply('light')
fn = dict((n, f) for n, _, f, _, _ in FIGURES)['05_pgd_recovery']
fig, rows = fn(inp, t)
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); plt.close(fig)
display(Image(buf.getvalue()))
print(f'{len(rows)} plotted values; first rows:'); [print(r) for r in rows[:4]]

### Figure 6. B-NORM catch rate

Share of frames with a sender in which B-NORM flags the attacker; on 'clean' this is the false-flag rate.

In [ ]:
t = style.apply('light')
fn = dict((n, f) for n, _, f, _, _ in FIGURES)['06_bnorm_catch']
fig, rows = fn(inp, t)
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); plt.close(fig)
display(Image(buf.getvalue()))
print(f'{len(rows)} plotted values; first rows:'); [print(r) for r in rows[:4]]

### Figure 7. Attack strength

Attack loss per condition over all frames (with CI) and over the frames where the attack was realized.

In [ ]:
t = style.apply('light')
fn = dict((n, f) for n, _, f, _, _ in FIGURES)['07_attack_strength']
fig, rows = fn(inp, t)
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); plt.close(fig)
display(Image(buf.getvalue()))
print(f'{len(rows)} plotted values; first rows:'); [print(r) for r in rows[:4]]

### Figure 8. B-RAY removals

Boxes B-RAY removed, split by whether they matched a real car.

In [ ]:
t = style.apply('light')
fn = dict((n, f) for n, _, f, _, _ in FIGURES)['08_bray_boxes']
fig, rows = fn(inp, t)
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); plt.close(fig)
display(Image(buf.getvalue()))
print(f'{len(rows)} plotted values; first rows:'); [print(r) for r in rows[:4]]

### Figure 9. R1 gating against ground truth

Contributions (sender × 6.4 m tile) R1 switched off, by their ground-truth label.

In [ ]:
t = style.apply('light')
fn = dict((n, f) for n, _, f, _, _ in FIGURES)['09_r1_units']
fig, rows = fn(inp, t)
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); plt.close(fig)
display(Image(buf.getvalue()))
print(f'{len(rows)} plotted values; first rows:'); [print(r) for r in rows[:4]]

### Figure 10. Effect by number of cars

Mean ΔAP@0.7 over the 4 original attacks per agent count (frozen v2 analysis, frame-order AP).

In [ ]:
t = style.apply('light')
fn = dict((n, f) for n, _, f, _, _ in FIGURES)['10_by_agents']
fig, rows = fn(inp, t)
buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=110, bbox_inches='tight'); plt.close(fig)
display(Image(buf.getvalue()))
print(f'{len(rows)} plotted values; first rows:'); [print(r) for r in rows[:4]]

## Per-condition tables

In [ ]:
for d in DEFS:
    display(Markdown(f'**{DEF_NAME[d]}**'))
    display(Markdown(T.per_family(inp, d)))

## PGD, B-NORM and descriptive expected-outcome checks

In [ ]:
if inp.pgd_conditions:
    display(Markdown(T.pgd(inp)))
    e = T.expected_outcome_checks(inp)
    display(Markdown(e or 'no operational checks in the preregistration'))
    n = T.d70_note(inp)
    display(Markdown(n)) if n else None

## Realized-only effects

In [ ]:
for d in DEFS:
    display(Markdown(f'**{DEF_NAME[d]}**'))
    display(Markdown(T.realized_only(inp, d)))

## Interpretation (AUTHOR)

*To be written after reading the cells above. Keep to what the numbers support; quote the pre-registered statements as emitted.*

## Run provenance

In [ ]:
import matplotlib, numpy, jinja2
print('python', platform.python_version(), '| numpy', numpy.__version__, '| matplotlib', matplotlib.__version__,
      '| jinja2', jinja2.__version__)
print('platform', platform.platform())
print('wall time %.1f s' % (time.time() - T0))